Corpus Search

Этот ноутбук позволяет искать по корпусу `dom.txt` различные элементы:
- по конкретному токену
- по лемме
- по семантическому тегу
- по комбинации семантических тегов

Измените значения переменных в соответствующих ячейках и выполните их, чтобы увидеть результаты.

In [2]:
# Подготовка: импорт и загрузка корпуса
import pymorphy3
import re
from collections import Counter
from pathlib import Path

In [5]:
# Путь к корпусу 
BASE_DIR = Path().resolve().parent
CORPUS_PATH = BASE_DIR / 'parse' / 'dom.txt'

In [6]:
# Чтение корпуса
with open(CORPUS_PATH, 'r', encoding='cp1251') as file:
    corpus = file.readlines()

In [7]:
# Функции для поиска
def extract_clean_text(line):
    return ' '.join([word.split('>')[1].split('<')[0]
                     for word in line.split() if '>' in word and '<' in word])

In [8]:
def count_token_frequency(token):
    return sum(1 for line in corpus if token in line)

In [9]:
def search_by_token(token, max_results=5):
    results = []
    for line in corpus:
        if token in line:
            results.append(extract_clean_text(line))
            if len(results) >= max_results:
                break
    freq = count_token_frequency(token)
    return freq, results

In [10]:
def count_lemma_frequency(lemma):
    morph = pymorphy3.MorphAnalyzer()
    cnt = 0
    for line in corpus:
        for word in line.split():
            clean = word.split('>')[1].split('<')[0] if '>' in word and '<' in word else word
            parsed = morph.parse(clean.strip('.,!?"\'"'))[0]
            if parsed.normal_form == lemma:
                cnt += 1
                break
    return cnt

In [11]:
def search_by_lemma(lemma, max_results=5):
    morph = pymorphy3.MorphAnalyzer()
    results = []
    for line in corpus:
        for word in line.split():
            clean = word.split('>')[1].split('<')[0] if '>' in word and '<' in word else word
            parsed = morph.parse(clean.strip('.,!?"\'"'))[0]
            if parsed.normal_form == lemma:
                results.append(extract_clean_text(line))
                break
        if len(results) >= max_results:
            break
    freq = count_lemma_frequency(lemma)
    return freq, results

In [12]:
def count_tag_frequency(tag):
    return sum(1 for line in corpus if tag in line)

In [13]:
def search_by_semantic_tag(tag, max_results=5):
    results = []
    pattern = re.compile(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]")
    for line in corpus:
        if pattern.search(line):
            clean = extract_clean_text(line)
            results.append(clean)
            if len(results) >= max_results:
                break
    freq = count_tag_frequency(tag)
    return freq, results

In [14]:
def count_tag_combination_frequency(tags):
    return sum(1 for line in corpus if all(tag in line for tag in tags))

In [15]:
def search_by_tag_combination(tags, max_results=5):
    results = []
    pattern = [re.compile(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]") for tag in tags]
    for line in corpus:
        if all(p.search(line) for p in pattern):
            context = extract_clean_text(line)
            results.append(context)
            if len(results) >= max_results:
                break
    freq = count_tag_combination_frequency(tags)
    return freq, results

In [16]:
# Пример: поиск по токену
token = 'рижском'  # замените на интересующий токен
freq, contexts = search_by_token(token, max_results=3)
print(f"Токен: {token}\nЧастота: {freq}\nКонтексты:")
for ctx in contexts:
    print("-", ctx)

Токен: рижском
Частота: 1
Контексты:
-   В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 


In [17]:
# Пример: поиск по лемме
lemma = 'рижский'  # замените на интересующую лемму
freq, contexts = search_by_lemma(lemma, max_results=3)
print(f"Лемма: {lemma}\nЧастота: {freq}\nКонтексты:")
for ctx in contexts:
    print("-", ctx)

Лемма: рижский
Частота: 2
Контексты:
-   В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 
-   В  июле  Кирилл   уехал  со  студенческим  отрядом  в  Новгород  а  мы  с  Ритой  в  конце  июля  взяли  путёвки  на  Рижское  взморье  поехали  немного  раньше  пожили  в  гостинице  а  с  августа      поселились  в доме отдыха 


In [18]:
# Пример: поиск по семантическому тегу
sem_tag = 'r:qual'  # замените на интересующий сем. тег
freq, contexts = search_by_semantic_tag(sem_tag, max_results=3)
print(f"Сем. тег: {sem_tag}\nЧастота: {freq}\nКонтексты:")
for ctx in contexts:
    print("-", ctx)

Сем. тег: r:qual
Частота: 1396
Контексты:
-   Добротный  деревянный дом  рядом  с  крыльцом  берёза  а  от дома дорога  ведёт  к  Богоявленской  церкви 
-   В  первой  половине  апреля  на  сцене  Балтийского дома  состоялся  6-й  международный  фестиваль  русских  театров  СНГ  и  Балтии  Встречи  в  России  собравший  в  Петербурге  спектакли  из  Белоруссии  и  Украины  Эстонии  Латвии  и  Армении 
-   В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 


In [19]:
# Пример: поиск по комбинации семант. тегов
tags = ['r:rel', 't:constr']  # замените на интересующую комбинацию
freq, contexts = search_by_tag_combination(tags, max_results=3)
print(f"Комбинация тегов: {', '.join(tags)}\nЧастота: {freq}\nКонтексты:")
for ctx in contexts:
    print("-", ctx)

Комбинация тегов: r:rel, t:constr
Частота: 1861
Контексты:
-   В  Кошкином доме  московского  театра  русской  драмы  Камерная  сцена  напротив  текст  произносился  и  пелся  пьеса  положена  на  музыку  А  Кулыгина  внятно  но  уж  слишком  назидательно  спектаклю  явственно  не  хватало  юмора
-   В  первой  половине  апреля  на  сцене  Балтийского дома  состоялся  6-й  международный  фестиваль  русских  театров  СНГ  и  Балтии  Встречи  в  России  собравший  в  Петербурге  спектакли  из  Белоруссии  и  Украины  Эстонии  Латвии  и  Армении 
-   В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 
